<a href="https://colab.research.google.com/github/DQN-Labs/Chess_AI/blob/main/DQN_Labs_Endpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DQN Labs AI - Free **endpoint** runtime
### Connect to the T4 runtime above and run the code below one by one.

In [ ]:
#@title Install

%cd /content/

!rm -rf build llama_bin.tar.gz
print("removed old build")

!wget -q https://huggingface.co/DQN-Labs/llamacpp-binaries-for-colab/resolve/main/llama_bin.tar.gz
print("got new build")

!tar -xzf llama_bin.tar.gz
print("extracted new build")

!ls build/bin | head -5

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pip install subprocess

import re
import subprocess

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\nURL for DQN Labs AI:\n{public_url}")
else:
    raise RuntimeError("❌ Could not find public Cloudflare URL.")


In [ ]:
#@title Select your model:
model_name = "dqnGPT-v1" #@param ["dqnGPT-v1", "dqnMath-v1", "dqnCode-v1", "dqnScience-v1"] {allow-input: true}

print(f"Using model: {model_name}")

def download_model(name):
    if name == "dqnGPT-v1":
        link = "https://huggingface.co/DQN-Labs-Community/dqnGPT-v1-GGUF/resolve/main/dqnGPT-v1.Q4_K_M.gguf"
        filename = "dqnGPT.gguf"

    elif name == "dqnMath-v1":
        link = "https://huggingface.co/DQN-Labs-Community/dqnMath-v1/resolve/main/DQN-Math-v1.Q4_K_M.gguf"
        filename = "dqnMath.gguf"

    elif name == "dqnCode-v1":
        link = "https://huggingface.co/DQN-Labs-Community/dqnCode-v1/resolve/main/DQN-Code-v1.Q4_K_M.gguf"
        filename = "dqnCode.gguf"

    elif name == "dqnScience-v1":
        print("Coming soon!")
        link = "https://huggingface.co/DQN-Labs-Community/dqnScience-v1-GGUF/resolve/main/DQN-Science-v1.Q4_K_M.gguf"
        filename = "dqnScience.gguf"

    else:
        print("Unknown model")
        return None

    print(f"Downloading {name}...")

    import os
    if os.path.exists(filename):
        print(f"{filename} already exists, skipping download")
    else:
        !wget -O {filename} {link}

    return filename


model_path = download_model(model_name)
print("Model ready at:", model_path)

## ⬇ Run this once the above is done, and copy the URL in the output to use the model!

In [ ]:
#@title 🚀 Start LLM Server

import subprocess
import time
import os

print("Please copy this URL and paste it as the endpoint in the website. This will stay here for 10 seconds:")
print(public_url)
time.sleep(10)

# Safety check
print("Selected model:", model_name)

# Start server based on model
if model_name == "dqnGPT-v1":
    assert os.path.exists("dqnGPT.gguf"), "Model not found, please try downloading the model again"

    !export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
    ./build/bin/llama-server -m dqnGPT.gguf -ngl 999 -c 16384 --port 8000 --no-webui


elif model_name == "dqnMath-v1":
    assert os.path.exists("dqnMath.gguf"), "Model not found, please try downloading the model again"

    !export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
    ./build/bin/llama-server -m dqnMath.gguf -ngl 999 -c 16384 --port 8000 --no-webui


elif model_name == "dqnCode-v1":
    assert os.path.exists("dqnCode.gguf"), "Model not found, please try downloading the model again"

    !export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
    ./build/bin/llama-server -m dqnCode.gguf -ngl 999 -c 16384 --port 8000 --no-webui


elif model_name == "dqnScience-v1":
    assert os.path.exists("dqnScience.gguf"), "Model not found, please try downloading the model again"

    !export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
    ./build/bin/llama-server -m dqnScience.gguf -ngl 999 -c 16384 --port 8000 --no-webui


else:
    print("Invalid model selection 💀")